# Ablation : les features dérivées de la date apportent-elles quelque chose ?

`heure`, `mois` et `jour_semaine` (dérivées de la date/heure du signalement) totalisent ensemble
**~1,4 % du gain total** du modèle `@staging` (`heure` 1,0 %, `jour_semaine` 0,3 %, `mois` 0,1 % —
importance native LightGBM, `booster_.feature_importance(importance_type="gain")`), contre 58,2 %
pour `departement` seul. Ce chiffre vient de l'importance native du modèle déjà entraîné, pas d'un
réentraînement sans ces features : ce notebook fait le vrai test d'ablation, même protocole que le
reste du benchmark (train 2019-2021, seuil calibré sur validation 2022, évalué sur holdout 2023).

Un seul modèle (LightGBM, la seule famille servable, cf. `ml/training/benchmark.py`), pas les 4
familles du benchmark complet : la question ici est l'apport de features sur le modèle déployé, pas
un nouveau comparatif de familles. Référence "avec date" : chiffres déjà publiés et validés de
`docs/ml_training_results.md` (config `"enriched"`, LightGBM, recall 0,808 / F1 macro 0,726,
seuil 0,47) — pas recalculés ici, comme documenté dans le même fichier pour éviter un réentraînement
redondant de plusieurs minutes.

Nécessite la stack dev démarrée (MLflow, PostgreSQL pour Gold, MinIO pour les artefacts) :
`docker compose -f infra/docker-compose.yml --env-file .env up -d`.

In [1]:
import mlflow
import sqlalchemy as sa

from gravia.config import get_settings
from ml.features.gold_features import (
    BASELINE_BOOLEAN_COLUMNS,
    BASELINE_CATEGORICAL_COLUMNS,
    BASELINE_NUMERIC_COLUMNS,
    ENRICHED_FLAG_COLUMNS,
    YEAR_COLUMN,
    load_gold_features,
    prepare_features,
    split_train_valid_test,
)
from ml.mlflow_env import configure_s3_artifact_env
from ml.training.benchmark import _to_pandas, evaluate_model

settings = get_settings()
configure_s3_artifact_env(settings)
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment("gravia-severity-classifier")
engine = sa.create_engine(settings.database.url)

DATE_FEATURES = ["heure", "mois", "jour_semaine"]

categorical = list(BASELINE_CATEGORICAL_COLUMNS)
other_enriched = list(BASELINE_NUMERIC_COLUMNS) + list(BASELINE_BOOLEAN_COLUMNS) + list(ENRICHED_FLAG_COLUMNS)
other_no_date = [c for c in other_enriched if c not in DATE_FEATURES]

print(f"Config 'enriched' (deployee) : {len(categorical) + len(other_enriched)} features")
print(f"Config 'enriched_no_date'    : {len(categorical) + len(other_no_date)} features (sans {DATE_FEATURES})")

Config 'enriched' (deployee) : 24 features
Config 'enriched_no_date'    : 21 features (sans ['heure', 'mois', 'jour_semaine'])


In [2]:
df = prepare_features(load_gold_features(engine))
train, valid, test = split_train_valid_test(df)
# Ordre chronologique requis par TimeSeriesSplit (cf. ml/training/benchmark.py::run_benchmark) :
# necessaire meme si mois/jour_semaine ne sont plus des features, ce sont ici des cles de tri.
train = train.sort([YEAR_COLUMN, "mois", "jour_semaine"])

x_train = _to_pandas(train, categorical, other_no_date)
x_valid = _to_pandas(valid, categorical, other_no_date)
x_test = _to_pandas(test, categorical, other_no_date)
y_train = train["is_grave"].to_pandas().astype(int)
y_valid = valid["is_grave"].to_pandas().astype(int)
y_test = test["is_grave"].to_pandas().astype(int)

print(f"train={len(x_train)}  valid={len(x_valid)}  test={len(x_test)}")

train=163102  valid=55302  test=54822


In [3]:
result = evaluate_model(
    "lightgbm",
    "enriched_no_date",  # etiquette de tracking MLflow, hors de l'enum FeatureSet officielle
    x_train, y_train, x_valid, y_valid, x_test, y_test,
    categorical, other_no_date,
)
print(
    f"Sans heure/mois/jour_semaine : recall={result['recall']:.3f}  "
    f"f1_macro={result['f1_macro']:.3f}  seuil={result['threshold']:.2f}  "
    f"seuils CDC {'atteints' if result['meets_gates'] else 'NON atteints'}"
)

2026/09/23 09:46:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/09/23 09:46:37 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason: MlflowException("Failed to enforce schema of data '  luminosite departement intersection meteo                type_collision  \\\n0          1          41            7     1     Deux véhicules - frontale   \n1          3          75            1     1  Deux véhicules - par le côté   \n2          1          21            1     2               Autre collision   \n3          1          22            9     1  Deux véhicules - par le côté   \n4          1          34            9     1  Deux véhicules - par le côté   \n\n  categorie_route regime_circulation voie_reservee profil_route trace_plan  \\\n0               4                  2             0            1          1   \n1               4                  2             0            1          1   \n2               4                  2             0            1          1   \n3               3                  2    

2026/09/23 09:46:48 WARNING mlflow.utils.requirements_utils: Failed to run predict on input_example, dependencies introduced in predict are not captured.
ValueError('train and valid dataset categorical_feature do not match.')Traceback (most recent call last):
  File "C:\Users\jackl\PycharmProjects\gravia\.venv\Lib\site-packages\mlflow\utils\_capture_modules.py", line 170, in load_model_and_predict
    model.predict(input_example, params=params)
  File "C:\Users\jackl\PycharmProjects\gravia\.venv\Lib\site-packages\mlflow\lightgbm\__init__.py", line 595, in predict
    return self.lgb_model.predict(dataframe)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\jackl\PycharmProjects\gravia\.venv\Lib\site-packages\lightgbm\sklearn.py", line 1597, in predict
    result = self.predict_proba(
             ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\jackl\PycharmProjects\gravia\.venv\Lib\site-packages\lightgbm\sklearn.py", line 1627, in predict_proba
    result = super().predict(
          

2026/09/23 09:46:49 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "l.... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.predict(model_uri=..., input_data=serving_input, env_manager="uv")` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: train and valid dataset categorical_feature do not match.


2026/09/23 09:46:49 INFO mlflow.models.model: Found the following environment variables used during model inference: [OPENAI_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


🏃 View run lightgbm-enriched_no_date at: http://localhost:5000/#/experiments/2/runs/8f1de5b31e0e4c8ea944a8dae1756f34
🧪 View experiment at: http://localhost:5000/#/experiments/2
Sans heure/mois/jour_semaine : recall=0.806  f1_macro=0.724  seuil=0.47  seuils CDC atteints


## Comparaison

| Configuration | Recall | F1 macro | Seuil |
|---|---|---|---|
| `enriched` (déployée, avec date) — référence publiée | 0,808 | 0,726 | 0,47 |
| `enriched_no_date` (sans `heure`/`mois`/`jour_semaine`) | **0,806** | **0,724** | 0,47 |

Écart : **-0,002 recall, -0,002 F1 macro** — du même ordre que le bruit déjà observé pour la
recherche d'hyperparamètres (docs/ml_training_results.md, «différence dans le bruit»). Les
deux seuils CDC restent atteints sans les 3 features dérivées de la date.

## Interprétation

Confirme empiriquement ce que l'importance native du modèle @staging suggérait déjà (heure 1,0 %,
jour_semaine 0,3 %, mois 0,1 % du gain total) : **les features dérivées de la date n'apportent rien
de mesurable** sur ce protocole. `departement` et les caractéristiques de l'accident (collision,
vitesse, véhicules impliqués) portent l'essentiel du signal ; `heure`/`mois`/`jour_semaine`
pourraient être retirées sans perte de performance réelle.

**Ne change rien au modèle déployé** : ce notebook ne réentraîne pas `@staging`, ne l'enregistre
pas dans le registry MLflow (pas d'appel à `register_best`) — c'est une mesure, pas un correctif.